# Building a Synthetic Image Dataset with Sana

_Authored by: [Parag Ekbote](https://github.com/ParagEkbote)_ and [David Berenstein](https://github.com/davidberenstein1957)

In [ ]:
!pip install -q pruna metaflow datasets

In [ ]:
import random
from pathlib import Path

import torch
from PIL import Image
from datasets import load_dataset
from collections import defaultdict
from transformers import AutoProcessor, AutoModelForImageTextToText
from diffusers import SanaPipeline
from pruna import smash, SmashConfig
from huggingface_hub import HfApi


In [ ]:
# Load the cleaned version of the dataset
dataset = load_dataset("data-is-better-together/open-image-preferences-v1", split="cleaned")

# Filter for preferred samples and valid simplified prompts
filtered_samples = []
for example in dataset:
    if example.get("preferred") in [0, 1] and example.get("simplified_prompt"):
        prompt_entry = {
            "prompt": example["simplified_prompt"].strip(),
            "category": example.get("category"),
            "subcategory": example.get("subcategory"),
        }
        filtered_samples.append(prompt_entry)


category_prompts = defaultdict(list)

for entry in filtered_samples:
    category_prompts[entry["category"]].append(entry["prompt"])

initial_prompts = []
for prompts_in_category in category_prompts.values():
    initial_prompts.extend(prompts_in_category[:5])  # max 5 from each category

# Limit to 50 prompts total
initial_prompts = initial_prompts[:50]


In [ ]:

adjectives = [
    "surreal", "epic", "dreamy", "vibrant", "macro",
    "eerie", "futuristic", "ancient", "minimalist", "baroque", "psychedelic"
]

modifiers = [
    "in the style of concept art",
    "with cinematic lighting",
    "as a digital painting",
    "with hyperrealistic textures",
    "as a low-poly render",
    "with bokeh background",
    "with watercolor effect"
]

def rewrite_prompts(prompt_list):
    rewritten = []
    for prompt in prompt_list:
        adjective = random.choice(adjectives)
        modifier = random.choice(modifiers)
        rewritten_prompt = f"{adjective} {prompt}, {modifier}"
        rewritten.append(rewritten_prompt)
    return rewritten


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pipeline = SanaPipeline.from_pretrained(
    "Efficient-Large-Model/Sana_600M_512px_diffusers",
    torch_dtype=torch.float16,
).to(device)

smash_config = SmashConfig()
smash_config['quantizer'] = 'hqq_diffusers'
smash_config['hqq_diffusers_weight_bits'] = 8

smashed_model = smash(
    model=pipeline,
    smash_config=smash_config,
)

def generate_images(prompt_list, iteration_number):
    output_folder = Path(f"outputs/iteration_{iteration_number}")
    output_folder.mkdir(parents=True, exist_ok=True)

    generated_pairs = []

    for index, prompt_text in enumerate(prompt_list[:50]):
        try:
            generated_image = smashed_model(prompt_text).images[0]
            image_file = output_folder / f"image_{iteration_number}_{index}.png"
            generated_image.save(image_file)
            generated_pairs.append((prompt_text, str(image_file)))
        except Exception as error:
            print(f"Generation failed for prompt '{prompt_text}': {error}")
    
    return generated_pairs


In [ ]:
vlm_processor = AutoProcessor.from_pretrained("google/gemma-3-4b-it")
vlm_model = AutoModelForImageTextToText.from_pretrained(
    "google/gemma-3-4b-it",
    torch_dtype=torch.float16 
).to(device)

def score_image(prompt_text, image_file_path):
    try:
        input_image = Image.open(image_file_path).convert("RGB")
        query = f"Does this image accurately represent the prompt: '{prompt_text}'? Answer only with 'yes' or 'no'."
        input_data = vlm_processor(images=input_image, text=query, return_tensors="pt").to(device)
        output_tokens = vlm_model.generate(**input_data, max_new_tokens=5)
        answer_text = vlm_processor.tokenizer.decode(output_tokens[0], skip_special_tokens=True).strip().lower()
        return 1.0 if "yes" in answer_text else 0.0
    except Exception as error:
        print(f"Scoring failed for prompt '{prompt_text}': {error}")
        return 0.0


In [ ]:
final_results = []
current_prompts = initial_prompts

for iteration_number in range(3):
    print(f"--- Iteration {iteration_number + 1} ---")
    updated_prompts = rewrite_prompts(current_prompts)
    generated_pairs = generate_images(updated_prompts, iteration_number)

    scored_iteration = []
    for prompt_text, image_path in generated_pairs:
        match_score = score_image(prompt_text, image_path)
        final_results.append((prompt_text, image_path, match_score, iteration_number))
        scored_iteration.append((prompt_text, image_path, match_score))

    scored_iteration.sort(key=lambda result: result[2], reverse=True)
    current_prompts = [entry[0] for entry in scored_iteration[:25]]


In [ ]:
sorted_by_score = sorted(final_results, key=lambda entry: entry[2], reverse=True)

print("\nTop 5 Generated Images by Score:")
for rank, (prompt_text, image_path, score, iteration_number) in enumerate(sorted_by_score[:5]):
    print(f"{rank+1}. [Iteration {iteration_number+1}] '{prompt_text}' → {image_path} [Score: {score:.2f}]")


In [ ]:
hub_api = HfApi()
account_info = hub_api.whoami()
print(f"Logged into Hugging Face as: {account_info['name']}")

prompt_list = [entry[0] for entry in final_results]
image_file_list = [entry[1] for entry in final_results]
score_list = [entry[2] for entry in final_results]
iteration_list = [entry[3] for entry in final_results]

data_for_hub = {
    "prompt": prompt_list,
    "image": [Image.open(path).convert("RGB") for path in image_file_list],
    "score": score_list,
    "iteration": iteration_list
}

image_dataset = Dataset.from_dict(data_for_hub)
image_dataset = image_dataset.cast_column("image", ImageFeature())
image_dataset.push_to_hub("your-username/synthetic-image-dataset", private=False)
